# Testes depois do treino 2: configurações externas à MarkerUNet

**Pergunta:** com recortes e o negativo do Cellpose, o pipeline estaciona em Dice ~0,835 (o Cellpose dá 0,845). Este notebook testa,
**sem retreinar** e com os modelos dos *folds* já gravados, configurações que ficam **fora da MarkerUNet**: o que entra no
ScribblePrompt e como a imagem inteira é segmentada. Cada fator muda **sozinho**, contra a base; no fim, uma condição junta os
vencedores.

| Fator | Valores testados (o do treino em negrito) | O que muda |
|---|---|---|
| Faixa livre do negativo do Cellpose | 1, 2, **3**, 5 px | quanto o negativo se aproxima das máscaras do Cellpose |
| τ do canal positivo | 0,3, 0,4, **0,5**, 0,6 | a partir de que valor o marcador vira "aqui há núcleo" para o ScribblePrompt |
| Sobreposição dos blocos | **1/4**, 1/2 | quantos blocos cobrem cada pixel na imagem inteira (25 ou 49 blocos) |
| Entrada do ScribblePrompt | **128²**, 160², 192² | a resolução em que o ScribblePrompt vê cada bloco de 256 px |
| TTA | **não**, sim | média das 8 rotações e espelhamentos da imagem, cada saída voltando à orientação original |

Cada uma das 37 imagens de treino é prevista pelo modelo do *fold* em que ela ficou de fora, com o limiar 0,5. A condição **base** é o
pipeline como foi treinado e precisa reproduzir o resultado registrado. **Ressalva:** o modelo foi treinado na configuração da base;
o que ganhar aqui precisa ser confirmado retreinando.

**Tudo é mostrado no notebook**; salvar é opcional (`SALVAR_RESULTADOS`). Use um ambiente com **GPU**.

## 0. Setup

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Pedro-io/cell-fuzzy-seg.git"
BRANCH = "homolog"


def find_repo_root(start="."):
    p = os.path.abspath(start)
    prev = None
    while p != prev:
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "pyproject.toml")):
            return p
        prev, p = p, os.path.dirname(p)
    return None


if "google.colab" in sys.modules:
    repo_root = os.path.join("/content", "cell-fuzzy-seg")
    if os.path.exists(repo_root):
        subprocess.run(["git", "-C", repo_root, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, repo_root], check=True)
else:
    repo_root = find_repo_root(os.getcwd())
    if repo_root is None:
        raise RuntimeError("Raiz do repositório não encontrada a partir do diretório atual.")

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Repositório:", repo_root)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
if "google.colab" in sys.modules:
    get_ipython().system("python -m pip install -r requirements.txt -q")
else:
    print("Ambiente local: verifique se as dependências do requirements.txt estão instaladas.")

## 1. Configuração (a mesma do experimento testado)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.data.load.preprocessed_dataset import NEGATIVE_KEY, load_preprocessed
from src.evaluation.figures import pick_cases, prediction_figure
from src.evaluation.metrics import binary_metrics
from src.evaluation.report import save_run, summarize
from src.losses.loss_composer import LossComposer
from src.losses.terms import DiceTerm
from src.models.configs.marker_unet_config import UNET_CONFIG
from src.models.networks.final_segmentation.scribble_prompting_network import ScribblePromptingNetwork
from src.models.networks.marker_unet import MarkerUNet
from src.pipeline.steps.inference.frozen_segmentation_step import FrozenSegmentationStep
from src.pipeline.steps.inference.marker_step import MarkerStep
from src.pipeline.training_pipeline import TrainingPipeline
from src.pipeline.tiled_pipeline import TiledPipeline
from src.training.checkpoints import fold_name, load_pipeline
from src.training.kfold import TrainConfig, final_epochs, run_kfold

# Experimento cujos modelos são testados: a configuração abaixo precisa ser a dele (a célula seguinte confere).
EXPERIMENT_NAME = "recortes_negativo"
OVERWRITE = False            # recusa gravar por cima de resultados ou modelos que já existem

RGBA_ALPHA = "mask"          # 4º canal da MarkerUNet: máscara do Cellpose ("prob": probabilidade)
TILE_SIZE = 256             # recorte do treino e bloco da avaliação (px da imagem original)
CROPS_PER_IMAGE = 4          # recortes sorteados de cada imagem por época
TILE_OVERLAP = 0.25          # sobreposição dos blocos na avaliação (fração do lado)
SP_INPUT_SIZE = (128, 128)   # entrada do ScribblePrompt (o recorte reduzido)
TAU = 0.5                    # limiar do canal positivo
NEGATIVE_RADIUS = 3              # None: sem negativo; n: negativo = pixels a mais de n px de qualquer máscara do Cellpose

CONFIG = TrainConfig(
    k=5, batch_size=16, max_epochs=200, patience=20, min_delta=0.0,
    lr=1e-4, grad_clip=1.0, threshold=0.5, seed=42, log_every=10,
    crop_size=TILE_SIZE, crops_per_image=CROPS_PER_IMAGE,
)
DESCRIPTION = {
    "perdas": "soft Dice (1,0), sem regularização do marcador",
    "rede_final": f"ScribblePrompt-UNet v1 congelado, scribble_mode=positive, tau={TAU}, input_size={SP_INPUT_SIZE}"
                  + ("" if NEGATIVE_RADIUS is None
                     else f", negativo = fundo do Cellpose (faixa livre de {NEGATIVE_RADIUS} px)"),
    "marker_net": "MarkerUNet (smp Unet, resnet34 ImageNet, 4 canais)",
    "rgba_alpha": RGBA_ALPHA,
    "aumentacao": "rot90 + flips, sorteada a cada época",
    "recortes": f"treino com {CROPS_PER_IMAGE} recortes de {TILE_SIZE}² por imagem por época (MarkerUNet sem redução); avaliação da imagem inteira em blocos de {TILE_SIZE}² com sobreposição {TILE_OVERLAP} e janela de Hann",
}

assert torch.cuda.is_available() or "google.colab" not in sys.modules, "No Colab, use um ambiente com GPU."
print("Device:", CONFIG.device)
print(json.dumps(DESCRIPTION, indent=2, ensure_ascii=False))

In [ ]:
SALVAR_RESULTADOS = False    # True: grava as tabelas em docs/estudo/resultados/testes_pos_treino_2_<experimento>/
RAIOS_NEGATIVO = [1, 2, 5]
TAUS = [0.3, 0.4, 0.6]
SOBREPOSICOES = [0.5]
ENTRADAS_SP = [160, 192]
IMAGENS_20X = {"TCGA-HE-7128-01Z-00-DX1", "TCGA-HE-7129-01Z-00-DX1", "TCGA-HE-7130-01Z-00-DX1"}

with open(os.path.join("docs", "estudo", "resultados", EXPERIMENT_NAME, "config.json"), encoding="utf-8") as f:
    REFERENCIA = json.load(f)
if REFERENCIA["descricao"] != DESCRIPTION:
    raise ValueError("A configuração deste notebook não é a do experimento testado; copie-a do notebook dele.")
print(f"Configuração de {EXPERIMENT_NAME} conferida.")

## 2. Onde estão os modelos

In [ ]:
# Modelos treinados: no Colab, no Google Drive de quem roda o notebook (sobrevive ao fim da sessão); fora dele,
# em ~/cell-fuzzy-models. Troque MODELS_ROOT para gravar em outro lugar.
if "google.colab" in sys.modules:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        MODELS_ROOT = "/content/drive/MyDrive/cell-fuzzy-models"
    except Exception as erro:
        print(f"Drive não montado ({erro}): os modelos ficam no disco da sessão e somem quando ela terminar.")
        MODELS_ROOT = "/content/cell-fuzzy-models"
else:
    MODELS_ROOT = os.path.expanduser(os.path.join("~", "cell-fuzzy-models"))
MODELS_DIR = os.path.join(MODELS_ROOT, EXPERIMENT_NAME)
print("Modelos em:", MODELS_DIR)

## 3. Dados de treino

In [ ]:
train_samples = load_preprocessed("train", alpha=RGBA_ALPHA, negative_radius=NEGATIVE_RADIUS)
TRAIN_IDS = sorted(train_samples)
print(f"Treino: {len(train_samples)} imagens (o teste não é carregado neste notebook)")

d = [binary_metrics(s["cellpose_segmentation"] > 0, s["ground_truth"])["dice"] for s in train_samples.values()]
print(f"Cellpose sozinho, treino: Dice {np.mean(d):.4f} ± {np.std(d, ddof=1):.4f}")

## 4. Rede final e montagem do experimento

In [ ]:
checkpoint = ScribblePromptingNetwork.download_checkpoint(os.path.join(MODELS_ROOT, "scribbleprompt"))
FINAL_NET = ScribblePromptingNetwork(
    checkpoint=checkpoint, device=CONFIG.device, scribble_mode="positive",
    positive_threshold=TAU, input_size=SP_INPUT_SIZE,
)


def factory():
    """Monta um experimento novo: MarkerUNet recém-inicializada, pipeline em blocos e perdas.

    O treino passa recortes de TILE_SIZE² (a MarkerUNet os vê sem redução); a avaliação passa a imagem
    inteira, que o TiledPipeline divide em blocos de TILE_SIZE² e cola de volta. O ScribblePrompt é
    congelado e não guarda estado, então a mesma instância serve a todos os folds.
    """
    marker_net = MarkerUNet(config=UNET_CONFIG)
    pipeline = TrainingPipeline([
        MarkerStep(model=marker_net, differentiable=True, target_size=TILE_SIZE, device=CONFIG.device),
        FrozenSegmentationStep(final_network=FINAL_NET, device=CONFIG.device,
                               negative_key=None if NEGATIVE_RADIUS is None else NEGATIVE_KEY),
    ])
    composer = LossComposer([DiceTerm()])
    return marker_net, TiledPipeline(pipeline, tile_size=TILE_SIZE, overlap=TILE_OVERLAP), composer

## 5. Modelos dos *folds* e funções das condições

Cada modelo traz, no `.json` ao lado, as imagens de validação do *fold*. As condições mudam, sem retreinar: o canal negativo
(recalculado com outra faixa livre), o τ e a entrada da rede final, a sobreposição dos blocos e o TTA.

In [ ]:
import time

import pandas as pd
from scipy.stats import wilcoxon

from src.data.load.preprocessed_dataset import PreprocessedDataset, cellpose_negative, collate_samples
from src.evaluation.metrics import binary_metrics
from src.evaluation.report import provenance

FOLDS = []
for k in range(CONFIG.k):
    pipeline, meta = load_pipeline(factory, MODELS_DIR, fold_name(k))
    FOLDS.append((k, pipeline, meta["val_ids"]))
print({k: len(ids) for k, _, ids in FOLDS}, "imagens de validação por fold")


def com_negativo(amostras, raio):
    """As mesmas amostras, com o canal negativo recalculado para outra faixa livre."""
    return {i: {**s, NEGATIVE_KEY: cellpose_negative(s["cellpose_segmentation"], raio)} for i, s in amostras.items()}


def com_sobreposicao(tiled, overlap):
    return TiledPipeline(tiled.pipeline, tile_size=tiled.tile_size, overlap=overlap)


def rodar(tiled, batch, tta):
    """Probabilidade (H, W) da imagem do batch; com TTA, média das 8 rotações e espelhamentos."""
    if not tta:
        return tiled.run(batch, verbose=False)["segmentation"][0, 0]
    soma = None
    for k in range(4):
        for espelhar in (False, True):
            def ida(x):
                if not (torch.is_tensor(x) and x.ndim == 4):
                    return x
                x = torch.rot90(x, k, dims=(-2, -1))
                return torch.flip(x, dims=(-1,)) if espelhar else x
            saida = tiled.run({c: ida(v) for c, v in batch.items()}, verbose=False)["segmentation"]
            if espelhar:
                saida = torch.flip(saida, dims=(-1,))
            saida = torch.rot90(saida, -k, dims=(-2, -1))
            soma = saida if soma is None else soma + saida
    return (soma / 8)[0, 0]


def avaliar(condicao, amostras=None, tau=TAU, entrada=SP_INPUT_SIZE, overlap=TILE_OVERLAP, tta=False):
    amostras = train_samples if amostras is None else amostras
    FINAL_NET.positive_threshold, FINAL_NET.input_size = tau, tuple(entrada)
    linhas = []
    try:
        for k, tiled, val_ids in FOLDS:
            tiled = tiled if overlap == TILE_OVERLAP else com_sobreposicao(tiled, overlap)
            for image_id in val_ids:
                batch = collate_samples([PreprocessedDataset(amostras, ids=[image_id])[0]])
                batch = {c: (v.to(CONFIG.device) if torch.is_tensor(v) else v) for c, v in batch.items()}
                with torch.no_grad():
                    prob = rodar(tiled, batch, tta).float().cpu().numpy()
                s = amostras[image_id]
                base = binary_metrics(s["cellpose_segmentation"] > 0, s["ground_truth"])
                linhas.append({"condicao": condicao, "id": image_id, "fold": k, "aumento_20x": image_id in IMAGENS_20X,
                               **binary_metrics(prob, s["ground_truth"]),
                               **{f"cellpose_{m}": v for m, v in base.items()}})
    finally:
        FINAL_NET.positive_threshold, FINAL_NET.input_size = TAU, tuple(SP_INPUT_SIZE)
    return linhas

## 6. Um fator por vez

In [ ]:
inicio = time.time()
LINHAS = avaliar("base")
for r in RAIOS_NEGATIVO:
    LINHAS += avaliar(f"faixa do negativo {r} px", amostras=com_negativo(train_samples, r))
for t in TAUS:
    LINHAS += avaliar(f"τ {t}", tau=t)
for o in SOBREPOSICOES:
    LINHAS += avaliar(f"sobreposição {o}", overlap=o)
for e in ENTRADAS_SP:
    LINHAS += avaliar(f"entrada do ScribblePrompt {e}²", entrada=(e, e))
LINHAS += avaliar("TTA", tta=True)
df = pd.DataFrame(LINHAS)
print(f"{(time.time() - inicio) / 60:.1f} min")


def comparar(df):
    base = df[df.condicao == "base"].set_index("id")
    tabela = []
    for condicao in dict.fromkeys(df.condicao):
        s = df[df.condicao == condicao].set_index("id").loc[base.index]
        d = s.dice - base.dice
        p = wilcoxon(s.dice, base.dice).pvalue if (d != 0).any() else float("nan")
        tabela.append({"condição": condicao, "Dice": s.dice.mean(), "precisão": s.precision.mean(),
                       "revocação": s.recall.mean(), "massa": s.mass_ratio.mean(), "Δ base": d.mean(),
                       "melhora/piora": f"{int((d > 0).sum())}/{int((d < 0).sum())}", "p": p,
                       "vence": bool(d.mean() > 0 and (d > 0).sum() > (d < 0).sum() and p < 0.05),
                       "acima do Cellpose": int((s.dice > s.cellpose_dice).sum()), "Dice 20×": s[s.aumento_20x].dice.mean()})
    return pd.DataFrame(tabela)


registro = pd.read_csv(os.path.join("docs", "estudo", "resultados", EXPERIMENT_NAME, "folds.csv"))
base = df[df.condicao == "base"]
print(f"Dice registrado na validação cruzada de {EXPERIMENT_NAME}: {registro.dice.mean():.4f} | base agora: {base.dice.mean():.4f}")
print(f"Cellpose: Dice {base.cellpose_dice.mean():.4f}, precisão {base.cellpose_precision.mean():.4f}, "
      f"revocação {base.cellpose_recall.mean():.4f}")
pd.set_option("display.width", 220)
tabela = comparar(df)
tabela.round(4)

### Como ler a tabela acima

- **Dice, precisão, revocação, massa:** médias das 37 imagens, cada uma prevista pelo modelo do *fold* em que ficou de fora, no
  limiar 0,5. A massa é o previsto dividido pelo GT (acima de 1, o pipeline segmenta demais).
- **Δ base:** a diferença média, imagem por imagem, contra a base; positiva = a condição ganhou. **melhora/piora** conta as
  imagens; **p** é o teste de Wilcoxon pareado.
- **vence:** Δ positivo, mais imagens melhorando que piorando e p < 0,05. É o critério para a condição entrar na combinação.
- **acima do Cellpose:** em quantas das 37 imagens a condição passa do Cellpose. **Dice 20×:** a média das três imagens em 20×.

## 7. Combinação dos vencedores

Para cada fator, o melhor valor que **venceu** a base (se nenhum venceu, fica o da base). A combinação é avaliada como mais uma
condição.

In [ ]:
def melhor(valores, padrao, formato):
    candidatos = [(v, tabela.set_index("condição").loc[formato(v)]) for v in valores]
    vencedores = [(v, linha["Dice"]) for v, linha in candidatos if linha["vence"]]
    return max(vencedores, key=lambda x: x[1])[0] if vencedores else padrao


raio = melhor(RAIOS_NEGATIVO, NEGATIVE_RADIUS, lambda r: f"faixa do negativo {r} px")
tau = melhor(TAUS, TAU, lambda t: f"τ {t}")
overlap = melhor(SOBREPOSICOES, TILE_OVERLAP, lambda o: f"sobreposição {o}")
entrada = melhor(ENTRADAS_SP, SP_INPUT_SIZE[0], lambda e: f"entrada do ScribblePrompt {e}²")
tta = bool(tabela.set_index("condição").loc["TTA", "vence"])
COMBO = f"combinação: faixa {raio} px, τ {tau}, sobreposição {overlap}, entrada {entrada}², TTA {'sim' if tta else 'não'}"
print(COMBO)
amostras = train_samples if raio == NEGATIVE_RADIUS else com_negativo(train_samples, raio)
df = pd.concat([df, pd.DataFrame(avaliar(COMBO, amostras=amostras, tau=tau, entrada=(entrada, entrada),
                                         overlap=overlap, tta=tta))], ignore_index=True)
tabela = comparar(df)
tabela.round(4)

## 8. Salvar (opcional)

In [ ]:
if SALVAR_RESULTADOS:
    pasta = os.path.join("docs", "estudo", "resultados", f"testes_pos_treino_2_{EXPERIMENT_NAME}")
    os.makedirs(pasta, exist_ok=False)
    df.to_csv(os.path.join(pasta, "por_imagem.csv"), index=False)
    tabela.to_csv(os.path.join(pasta, "resumo.csv"), index=False)
    with open(os.path.join(pasta, "config.json"), "w", encoding="utf-8") as f:
        json.dump({"experimento": EXPERIMENT_NAME, "raios_negativo": RAIOS_NEGATIVO, "taus": TAUS,
                   "sobreposicoes": SOBREPOSICOES, "entradas_sp": ENTRADAS_SP, "combinacao": COMBO,
                   "procedencia": provenance()}, f, indent=2, ensure_ascii=False)
    print("Resultados em:", pasta)
else:
    print("Nada salvo (SALVAR_RESULTADOS = False).")